## Laboratorio Neo4j - CRUD y Algoritmos de Grafos

## Sección 1 - Creación de la conexión

Edita tu URI y credenciales de conexión en la celda de abajo. Usaremos el driver oficial de Python `neo4j`.

In [1]:
# BORRA ESTE COMENTARIO Y LA EXCEPCION DE ABAJO Y PON TU CODIGO AQUI
neo4j_uri = "bolt://localhost:7687"
neo4j_user = "neo4j"
neo4j_password = "password"  

In [2]:
from neo4j import GraphDatabase

try:
    driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))
    driver.verify_connectivity()
    print("✅ Conexión exitosa a Neo4j")
except Exception as e:
    print(f"❌ Error conectando a Neo4j: {e}")

✅ Conexión exitosa a Neo4j


## Sección 2 - Inserción de datos (CREATE)

Vamos a inicializar nuestra base de datos con el ecosistema de *GraphStore*. 

Utiliza el método `session.execute_write()` para ejecutar una consulta Cypher que inserte el siguiente lote de datos usando `UNWIND` y `MERGE`:

```json
[
  {cliente: "Alice", producto: "Laptop", tarjeta: "1111-2222"},
  {cliente: "Bob", producto: "Laptop", tarjeta: "3333-4444"},
  {cliente: "Bob", producto: "Mouse", tarjeta: "3333-4444"},
  {cliente: "Charlie", producto: "Mouse", tarjeta: "1111-2222"},
  {cliente: "Diana", producto: "Teclado", tarjeta: "5555-6666"},
  {cliente: "Alice", producto: "Audífonos", tarjeta: "1111-2222"}
]
```

Asegúrate de crear los nodos `Usuario`, `Producto`, `Tarjeta` y las relaciones `[:COMPRÓ]` y `[:USA]`.

In [3]:
# BORRA ESTE COMENTARIO Y LA EXCEPCION DE ABAJO Y PON TU CODIGO AQUI
lote = [
    {"cliente": "Alice",   "producto": "Laptop",    "tarjeta": "1111-2222"},
    {"cliente": "Bob",     "producto": "Laptop",    "tarjeta": "3333-4444"},
    {"cliente": "Bob",     "producto": "Mouse",     "tarjeta": "3333-4444"},
    {"cliente": "Charlie", "producto": "Mouse",     "tarjeta": "1111-2222"},
    {"cliente": "Diana",   "producto": "Teclado",   "tarjeta": "5555-6666"},
    {"cliente": "Alice",   "producto": "Audífonos", "tarjeta": "1111-2222"},
]

def insertar_lote(tx, lote):
    query = """
    UNWIND $lote AS fila
    MERGE (u:Usuario {nombre: fila.cliente})
    MERGE (p:Producto {nombre: fila.producto})
    MERGE (t:Tarjeta {numero: fila.tarjeta})
    MERGE (u)-[:COMPRÓ]->(p)
    MERGE (u)-[:USA]->(t)
    """
    tx.run(query, lote=lote)

with driver.session() as session:
    session.execute_write(insertar_lote, lote)

In [4]:
print("Validando grafo insertado...")

Validando grafo insertado...


## Sección 3 - Consultas (READ)

Escribe una consulta Cypher para encontrar todos los productos que compró "Bob".

***Almacena el resultado (una lista con los nombres de los productos) en una variable llamada `productos_bob`***

In [5]:
productos_bob = []

with driver.session() as session:
    result = session.run("""
        MATCH (:Usuario {nombre: 'Bob'})-[:COMPRÓ]->(p:Producto)
        RETURN p.nombre AS producto
    """)
    productos_bob = [record["producto"] for record in result]

print(productos_bob)

['Mouse', 'Laptop']


In [6]:
print('Validando resultado de la consulta en la variable "productos_bob"...')

Validando resultado de la consulta en la variable "productos_bob"...


## Sección 4 - Actualización (UPDATE)

Diana ha reportado que perdió su tarjeta y el banco le ha enviado una nueva.
Actualiza el nodo de la tarjeta que Diana `[:USA]` para que su número ahora sea `'9999-9999'`.

In [7]:
with driver.session() as session:
    session.run("""
        MATCH (:Usuario {nombre: 'Diana'})-[:USA]->(t:Tarjeta)
        SET t.numero = '9999-9999'
    """)

In [8]:
print("Validando actualización...")

Validando actualización...


## Sección 5 - El Poder de los Grafos: Algoritmos y Patrones

Aquí es donde Neo4j y Cypher demuestran su superioridad sobre las bases de datos tabulares. Vamos a resolver problemas complejos de conectividad.

### 5.1 Motor de Recomendación Colaborativo

Alice acaba de entrar a la tienda. Busca qué otros usuarios compraron los mismos productos que Alice y **recomiéndale** los productos que esos otros usuarios compraron, pero que Alice aún NO tiene.

***Guarda el nombre del producto recomendado en una variable llamada `recomendacion_alice`***

In [9]:
recomendacion_alice = None

with driver.session() as session:
    result = session.run("""
        MATCH (:Usuario {nombre: 'Alice'})-[:COMPRÓ]->(p1:Producto)<-[:COMPRÓ]-(otro:Usuario)
        WHERE otro.nombre <> 'Alice'
        MATCH (otro)-[:COMPRÓ]->(p2:Producto)
        WHERE NOT EXISTS {
            MATCH (:Usuario {nombre: 'Alice'})-[:COMPRÓ]->(p2)
        }
        RETURN p2.nombre AS recomendacion
        LIMIT 1
    """)
    
    record = result.single()
    if record:
        recomendacion_alice = record["recomendacion"]

print(f"Recomendación para Alice: {recomendacion_alice}")

Recomendación para Alice: Mouse


In [10]:
print("Validando el motor de recomendaciones...")

Validando el motor de recomendaciones...


### 5.2 Búsqueda de Rutas (Shortest Path)

¿A cuántos 'grados de separación' están conectados Diana y Charlie? Utiliza la función `shortestPath()` para encontrar el camino más corto entre ellos sin importar el tipo de relación o la dirección de la flecha.

***Almacena el número de saltos (longitud del camino) en una variable `distancia_diana_charlie`.***

In [11]:
distancia_diana_charlie = 0

with driver.session() as session:
    result = session.run("""
        MATCH (d:Usuario {nombre: 'Diana'}), (c:Usuario {nombre: 'Charlie'})
        OPTIONAL MATCH p = shortestPath((d)-[*]-(c))
        RETURN coalesce(length(p), 0) AS distancia
    """)
    
    record = result.single()
    if record:
        distancia_diana_charlie = record["distancia"]

print(f"Grados de separación: {distancia_diana_charlie}")

Grados de separación: 0


In [12]:
print("Validando cálculo de ruta más corta...")

Validando cálculo de ruta más corta...


### 5.3 Detección de Fraude (Patrones de Anomalía)

El equipo de seguridad sospecha de tarjetas clonadas. Encuentra si existen **dos usuarios diferentes** que estén compartiendo exactamente la **misma tarjeta de crédito** en el sistema.

***Almacena el resultado (una lista de tuplas con los nombres de los usuarios sospechosos, ej: `[('Alice', 'Charlie')]`) en la variable `usuarios_sospechosos`.***
*Nota: Para evitar duplicados como (A, B) y (B, A), recuerda usar `id(u1) < id(u2)` o comparar sus nombres alfabéticamente.*

In [13]:
usuarios_sospechosos = []

with driver.session() as session:
    result = session.run("""
        MATCH (u1:Usuario)-[:USA]->(t:Tarjeta)<-[:USA]-(u2:Usuario)
        WHERE u1.nombre < u2.nombre
        RETURN u1.nombre AS usuario1, u2.nombre AS usuario2
    """)
    
    usuarios_sospechosos = [
        (record["usuario1"], record["usuario2"])
        for record in result
    ]

print(f"Usuarios sospechosos encontrados: {usuarios_sospechosos}")

Usuarios sospechosos encontrados: [('Alice', 'Charlie')]


In [14]:
print("Validando algoritmo de detección de fraude...")

Validando algoritmo de detección de fraude...


## Sección 6 - Eliminación de datos (DELETE)

Alice ha decidido devolver los 'Audífonos' a la tienda. Escribe una consulta para eliminar **únicamente la relación** `[:COMPRÓ]` entre Alice y los Audífonos, sin borrar los nodos de Usuario ni de Producto.

***Guarda el valor `True` en la variable `relacion_eliminada` si el código se ejecuta sin errores.***

In [15]:
relacion_eliminada = False

with driver.session() as session:
    session.run("""
        MATCH (:Usuario {nombre: 'Alice'})-[r:COMPRÓ]->(:Producto {nombre: 'Audífonos'})
        DELETE r
    """)
    relacion_eliminada = True
    
print(f"Relación eliminada: {relacion_eliminada}")

Relación eliminada: True


In [16]:
print("Validando eliminación de relación...")

Validando eliminación de relación...


## Sección 7 - Agrupación y Ordenamiento (WITH, ORDER BY)

¿Cuáles son los productos más populares en la tienda? Cuenta el número de usuarios que compraron cada producto y filtra para obtener los productos que tienen **más de 1 venta**.

***Guarda el resultado (una lista con los nombres de los productos) en `productos_populares`.***

In [17]:
productos_populares = []

with driver.session() as session:
    result = session.run("""
        MATCH (:Usuario)-[:COMPRÓ]->(p:Producto)
        WITH p, count(*) AS ventas
        WHERE ventas > 1
        RETURN p.nombre AS producto
        ORDER BY ventas DESC, producto ASC
    """)
    
    productos_populares = [record["producto"] for record in result]
    
print(f"Productos con más de 1 venta: {productos_populares}")

Productos con más de 1 venta: ['Laptop', 'Mouse']


In [18]:
print("Validando agrupación y ordenamiento...")

Validando agrupación y ordenamiento...


## Sección 8 - Filtrado por Patrones Indirectos (Co-compras)

Queremos sugerir accesorios directamente en la página de ventas de la 'Laptop'. Encuentra qué *otros productos* compraron los usuarios que también adquirieron una 'Laptop' (excluyendo la 'Laptop' en los resultados finales).

***Guarda el resultado (una lista con los nombres de estos productos) en `productos_co_comprados`.***

In [20]:
productos_co_comprados = []

with driver.session() as session:
    result = session.run("""
        MATCH (:Producto {nombre: 'Laptop'})<-[:COMPRÓ]-(u:Usuario)-[:COMPRÓ]->(p:Producto)
        WHERE p.nombre <> 'Laptop'
        RETURN DISTINCT p.nombre AS producto
        ORDER BY producto
    """)
    
    productos_co_comprados = [record["producto"] for record in result]

print(f"Quienes compraron Laptop también compraron: {productos_co_comprados}")

Quienes compraron Laptop también compraron: ['Mouse']


In [21]:
print("Validando patrones indirectos...")

Validando patrones indirectos...


## Sección 9 - Centralidad de Grado (Degree Centrality)

Encuentra al usuario que ha realizado la mayor cantidad de compras en toda la tienda (el nodo de tipo Usuario con más relaciones salientes de tipo `[:COMPRÓ]`).

***Guarda el nombre de este usuario en la variable de texto `usuario_mas_activo`.***

In [22]:
usuario_mas_activo = ""

with driver.session() as session:
    result = session.run("""
        MATCH (u:Usuario)-[:COMPRÓ]->(:Producto)
        WITH u, count(*) AS total_compras
        RETURN u.nombre AS usuario
        ORDER BY total_compras DESC, usuario ASC
        LIMIT 1
    """)
    
    record = result.single()
    if record:
        usuario_mas_activo = record["usuario"]
        
print(f"El usuario más activo es: {usuario_mas_activo}")

El usuario más activo es: Bob


In [23]:
print("Validando algoritmo de centralidad...")

Validando algoritmo de centralidad...
